## Assignment 3: Search and Games

Welcome to the third assignment of CS 541! In Assignments 1 and 2 the machine learned from data. Here nothing is learned: the agent *searches*. You will play tic-tac-toe against a rule-based bot, and then build a sequence of stronger players, from a one-move greedy agent up to full minimax with alpha-beta pruning, until you have one that provably never loses. This assignment must be done individually.

After this assignment, you should be able to:
1. Formalize a two-player game as a search problem, and evaluate agents by playing matches between them.  
2. Implement a greedy agent, minimax, alpha-beta pruning, and depth-limited search with an evaluation function, and measure what each costs.  

In CS541, any work generated by an AI shouldn't be included without declaration. If you include material generated by an AI, the level of AI use should be properly documented, and the actual tool should be noted (e.g., "I used Codex to proofread the codes and draft the analysis").

Nothing in this assignment is downloaded. Part 1 uses `ipywidgets` to draw a clickable board; if the import below fails, run `pip install ipywidgets` in your environment and restart the kernel. Everything else is plain Python plus numpy, pandas and matplotlib. Run the cell below first.

In [ ]:
import math
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 1. Play the game (6')
Tic-tac-toe is written below exactly as the lecture formalizes a game: an initial state $S_0$, $\text{Player}(s)$, $\text{Actions}(s)$, $\text{Result}(s, a)$, $\text{Terminal-Test}(s)$ and $\text{Utility}(s, p)$. A state is a tuple of 9 characters, `"X"`, `"O"` or `"."`, with cells numbered 0 to 8 row by row. X always moves first. An **agent** is any function that takes a state and returns the cell it wants to play. Everything in this part and the next two is built from that one interface.

Two agents are given. `RandomAgent(seed)` plays uniformly at random among the legal moves, using its own seeded generator, so a match against it is reproducible. `rule_based_agent` follows five rules in order: (1) win now if it can; (2) otherwise block the opponent's immediate win; (3) otherwise take the center; (4) otherwise take the first free corner in the order 0, 2, 6, 8; (5) otherwise take the first free edge in the order 1, 3, 5, 7. It is deterministic.

1.1 Complete `play_game()` and `run_match()` below.  
1.2 Play the bot with the widget: at least **five** complete games, at least **two** as X and at least **two** as O. The widget prints the full move list of every game when it ends. Paste those lists into `my_games`, one list per game.  
1.3 Find a sequence of your own moves, as X, that beats the bot. Store *your moves only* as `beat_the_bot_moves`. Then explain which of the bot's five rules let it down, and why no rule of that form could have saved it.  
1.4 Run three baseline matches of 1000 games each and store the result dicts as `match_random_random` (`RandomAgent(0)` as X, `RandomAgent(1)` as O), `match_rule_vs_random` (`rule_based_agent` as X, `RandomAgent(1)` as O) and `match_random_vs_rule` (`RandomAgent(0)` as X, `rule_based_agent` as O). Comment on your observations. The bot loses a handful of games as O to a *random* player: find one of those games and say which rule failed.  

Following are some specifications that may be helpful:
- `play_game(agent_x, agent_o)` always starts from `initial_state()`, asks the agent whose turn it is for exactly one move per ply, and returns `(winner, moves)`, where `winner` is `"X"`, `"O"` or `None` for a draw, and `moves` is the list of every cell played, in order.  
- `run_match(agent_x, agent_o, n_games)` returns a dict with exactly the keys `"X"`, `"O"` and `"draw"`, counting games.  
- Create each `RandomAgent` once, right before its match, and pass the same object into all of that match's games. Its generator then advances from game to game, and the whole match is reproducible.  
- Note: don't modify the given engine cell. Every later part, and the grader, relies on it behaving exactly as written.  

In [ ]:
# Given -- do not modify. The game, in the lecture's formalization.
EMPTY = "."
LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8),   # rows
         (0, 3, 6), (1, 4, 7), (2, 5, 8),   # columns
         (0, 4, 8), (2, 4, 6)]              # diagonals


def initial_state():
    """S_0: the empty board."""
    return (EMPTY,) * 9


def player(s):
    """Player(s): X moves first, so it is X's turn whenever the counts are equal."""
    return "X" if s.count("X") == s.count("O") else "O"


def other(p):
    return "O" if p == "X" else "X"


def actions(s):
    """Actions(s): the empty cells, in increasing order."""
    return [i for i, c in enumerate(s) if c == EMPTY]


def result(s, a):
    """Result(s, a): a NEW state with the current player's mark on cell a."""
    if s[a] != EMPTY:
        raise ValueError("cell {} is already taken".format(a))
    return s[:a] + (player(s),) + s[a + 1:]


def winner(s):
    """'X' or 'O' if that player has three in a line, else None."""
    for i, j, k in LINES:
        if s[i] != EMPTY and s[i] == s[j] == s[k]:
            return s[i]
    return None


def terminal(s):
    """Terminal-Test(s): someone has won, or the board is full."""
    return winner(s) is not None or EMPTY not in s


def utility(s, p):
    """Utility(s, p) at a terminal state: +1 if p won, -1 if p lost, 0 for a draw."""
    w = winner(s)
    return 0 if w is None else (1 if w == p else -1)


def show_board(s):
    print("\n".join(" " + " | ".join(s[r * 3:r * 3 + 3]) for r in range(3)))


def all_reachable_states():
    """Every board reachable from the empty board by legal play, terminal ones included."""
    seen, stack = set(), [initial_state()]
    while stack:
        s = stack.pop()
        if s in seen:
            continue
        seen.add(s)
        if not terminal(s):
            stack.extend(result(s, a) for a in actions(s))
    return seen


class RandomAgent:
    """Plays uniformly at random among the legal moves, from its own seeded generator."""
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)

    def __call__(self, s):
        acts = actions(s)
        return acts[self.rng.integers(len(acts))]


CORNERS, EDGES = (0, 2, 6, 8), (1, 3, 5, 7)


def rule_based_agent(s):
    """Win if possible, else block, else center, else first free corner, else first free edge."""
    me, opp = player(s), other(player(s))
    acts = actions(s)
    for a in acts:                                   # rule 1: win now
        if winner(result(s, a)) == me:
            return a
    for a in acts:                                   # rule 2: block the opponent's win
        if winner(s[:a] + (opp,) + s[a + 1:]) == opp:
            return a
    if s[4] == EMPTY:                                # rule 3: center
        return 4
    for a in CORNERS + EDGES:                        # rules 4 and 5: corners, then edges
        if s[a] == EMPTY:
            return a


print(len(all_reachable_states()), "reachable boards")
show_board(result(result(initial_state(), 4), 0))

In [ ]:
# Starter
def play_game(agent_x, agent_o):
    """Play one game from the empty board. Returns (winner, moves)."""
    state = initial_state()
    moves = []
    # TODO -- until the game is over: ask the agent whose turn it is for a move,
    # apply it with result(), and record it in moves
    return winner(state), moves


def run_match(agent_x, agent_o, n_games):
    """Play n_games and return {"X": x_wins, "O": o_wins, "draw": draws}."""
    counts = {"X": 0, "O": 0, "draw": 0}
    # TODO
    return counts


# a quick check: two copies of the deterministic bot always play the same game
print(play_game(rule_based_agent, rule_based_agent))

#### The widget
Run the cell below and click a square to move; the bot answers immediately. Use the two buttons on top to start a new game as X or as O. When a game ends, its full move list (your moves and the bot's) is printed under the board: copy those lists into `my_games` in the next cell. The lists are only printed, never saved, so copy them before you restart the kernel.

In [ ]:
# Given -- do not modify.
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    print("ipywidgets is not installed: run `pip install ipywidgets`, then restart the kernel.")


class TicTacToeWidget:
    """A clickable board against `bot`. Every finished game is printed and kept in self.history."""
    def __init__(self, bot):
        self.bot, self.history = bot, []
        cell = widgets.Layout(width="64px", height="64px")
        self.buttons = [widgets.Button(description=" ", layout=cell) for _ in range(9)]
        for i, b in enumerate(self.buttons):
            b.style.font_weight = "bold"
            b.on_click(lambda _, i=i: self._human_move(i))
        as_x = widgets.Button(description="New game: I play X")
        as_o = widgets.Button(description="New game: I play O")
        as_x.on_click(lambda _: self.new_game("X"))
        as_o.on_click(lambda _: self.new_game("O"))
        grid = widgets.GridBox(self.buttons, layout=widgets.Layout(
            grid_template_columns="repeat(3, 64px)", grid_gap="4px"))
        self.status, self.log = widgets.HTML(), widgets.Output()
        self.ui = widgets.VBox([widgets.HBox([as_x, as_o]), grid, self.status, self.log])
        self.new_game("X")

    def new_game(self, human):
        self.human, self.state, self.moves = human, initial_state(), []
        if human == "O":
            self._apply(self.bot(self.state))
        self._render()

    def _apply(self, a):
        self.state = result(self.state, a)
        self.moves.append(a)

    def _human_move(self, i):
        if terminal(self.state) or player(self.state) != self.human or self.state[i] != EMPTY:
            return
        self._apply(i)
        if not terminal(self.state):
            self._apply(self.bot(self.state))
        self._render()

    def _render(self):
        over = terminal(self.state)
        for i, b in enumerate(self.buttons):
            b.description = " " if self.state[i] == EMPTY else self.state[i]
            b.disabled = over or self.state[i] != EMPTY
        if not over:
            self.status.value = "You are <b>{}</b>. Your move.".format(self.human)
            return
        w = winner(self.state)
        outcome = "draw" if w is None else ("you win" if w == self.human else "the bot wins")
        self.status.value = "<b>Game over: {}.</b>".format(outcome)
        self.history.append(list(self.moves))
        with self.log:
            print("you played {}, {}: {}".format(self.human, outcome, self.moves))


if widgets is not None:
    game = TicTacToeWidget(rule_based_agent)
    display(game.ui)

In [ ]:
# TODO -- 1.2 paste the move lists the widget printed: at least five games,
# at least two as X and at least two as O
my_games = [
    # [4, 0, ...],
]

# TODO -- 1.3 YOUR moves only, as X, in order, in a game the bot loses
beat_the_bot_moves = None

# TODO -- 1.4 the three baseline matches, 1000 games each
match_random_random = None
match_rule_vs_random = None
match_random_vs_rule = None

*Your answers for 1.3 and 1.4:*

### 2. A greedy agent (7')
The rule-based bot hard-codes what to do. A *greedy* agent is the simplest thing that searches instead: it tries each legal move, scores the board it would produce with an evaluation function, and plays the best one. This is greedy best-first search from Lecture 4, applied to a game: it ranks moves by a heuristic alone and never looks at what happens next.
2.1 Implement `evaluate(s, p)`, the line-counting evaluation function defined below.  
2.2 Implement `greedy_agent(s)`.  
2.3 Build a dict named `greedy_results` with exactly the keys `"greedy_vs_random"`, `"random_vs_greedy"`, `"greedy_vs_rule"` and `"rule_vs_greedy"`, each holding a `run_match` result. The first-named agent plays X. Use 1000 games and `RandomAgent(1)` as O / `RandomAgent(0)` as X for the random matches, and a single game for the two matches against the bot. Briefly explain why one game is enough there.  
2.4 Find a non-terminal state where `greedy_agent` plays a move that is strictly worse, under perfect play, than the best move available, and store it as `greedy_trap_state`. It must be a different board from the fork position `X . . / . O . / . . X` (O to move) from 1.3. You will need Part 3's minimax to verify it, so come back to this after 3.1. Then write your analysis.  

The evaluation function scores a board for player $p$ as follows. If someone has won, it is $+1000$ if $p$ won and $-1000$ if $p$ lost. Otherwise, sum over the 8 lines (3 rows, 3 columns, 2 diagonals): a line holding $k \ge 1$ of $p$'s marks and none of the opponent's scores $+1$ for $k=1$ and $+10$ for $k=2$; a line holding the opponent's marks and none of $p$'s scores $-1$ or $-10$ the same way; an empty or mixed line scores 0. For example, on the fork board `X . . / . O . / . . X`, O owns three lines with one mark each and X owns four, so `evaluate(s, "O")` is $3 - 4 = -1$.

Following are some specifications that may be helpful:
- `evaluate(s, p)` is from player `p`'s viewpoint, so `evaluate(s, "O")` should always equal `-evaluate(s, "X")`. Use the constants given in the starter.  
- `greedy_agent(s)` scores each child `result(s, a)` from the viewpoint of `player(s)`, the player about to move, and returns the action with the highest score. Break ties toward the lowest cell index: loop over `actions(s)` in order and replace the best only on a strictly higher score.  
- A winning move scores $+1000$, far above anything a non-terminal board can reach, so greedy always takes an immediate win. Notice what it does *not* always do.  

In [ ]:
# Starter
LINE_WEIGHTS = {1: 1, 2: 10}   # a line with k of one player's marks and none of the other's
WIN_SCORE = 1000


def evaluate(s, p):
    """Heuristic value of board s for player p. +-WIN_SCORE if someone has won."""
    # TODO -- the terminal case first, then the sum over LINES
    return 0


def greedy_agent(s):
    """One-ply lookahead: the move whose resulting board evaluate() scores highest for the mover."""
    # TODO
    return None


# TODO -- 2.3
greedy_results = None

# TODO -- 2.4 (after Part 3.1): a board where greedy's move is worse than the best move
greedy_trap_state = None

*Your answer for 2.4:*

### 3. Minimax and alpha-beta (12')
Now search to the bottom of the tree. Throughout this part X is MAX and O is MIN, and every value is from **X's viewpoint**, $\text{Utility}(s, \text{X}) \in \{-1, 0, +1\}$.
3.1 Implement `minimax(s, stats)`, the lecture's recursion. Store the value of the empty board as `value_empty` and the number of nodes visited as `nodes_minimax`. Then implement `minimax_agent(s)` with the given `best_action()` helper.  
3.2 Implement `alphabeta(s, alpha, beta, stats, order)` with the lecture's pruning rule, and `order_moves(s)`, which returns the legal moves with the center first, then corners, then edges. Store the nodes visited from the empty board as `nodes_ab_natural` (the default `actions()` order) and `nodes_ab_ordered` (with `order=order_moves`). Then check `alphabeta` against `minimax` on every board from `all_reachable_states()` and store the number of disagreements as `ab_mismatches`.  
3.3 Implement `depth_limited(s, depth, ...)`, alpha-beta that stops after `depth` plies and scores the frontier with `evaluate(s, "X")`, and `make_depth_limited_agent(depth)`. Store as `depth1_agreement` the fraction of non-terminal reachable boards on which `make_depth_limited_agent(1)` and `greedy_agent` choose the same move. Then build `depth_sweep_df` with exactly the columns `["depth", "nodes_first_move", "wins", "draws", "losses"]` for depths 1 to 6, and plot wins and losses against depth.  
3.4 Write your analysis.  

Following are some specifications that may be helpful:
- `stats` is a dict such as `{"nodes": 0}`. Every function in this part adds 1 to `stats["nodes"]` at the top of **every call**, terminal or not, root included, before any pruning check. Skip the counting when `stats` is `None`. With that convention, `minimax` visits exactly 549,946 nodes from the empty board.  
- In `alphabeta`, a MAX node returns as soon as its running value is $\ge \beta$, and a MIN node as soon as its running value is $\le \alpha$. Update $\alpha$ (at MAX) or $\beta$ (at MIN) after each child. `order` is `None` or a function from a state to its moves.  
- `best_action(s, value_fn)` is given: `value_fn` maps a child state to its value from X's viewpoint, and the helper picks the highest for X and the lowest for O, ties to the lowest cell. Use it for every agent in this part, so that all agents break ties the same way.  
- `make_depth_limited_agent(depth)` counts the agent's own move as the first ply: it scores each child with `depth_limited(child, depth - 1)`. With that convention, depth 1 is exactly the greedy agent, which is what `depth1_agreement` checks.  
- For `depth_sweep_df`, play each depth-$d$ agent for 200 games as X against `RandomAgent(1)` and 200 games as O against `RandomAgent(2)`, creating fresh random agents for every depth, and sum the two matches from the depth-$d$ agent's viewpoint. `nodes_first_move` is the node count of `depth_limited(initial_state(), depth)`.  
- A full `minimax` from the empty board takes a second or two. `minimax_agent` from the empty board does it again for every move, so don't play whole matches with it here.  

In [ ]:
# Starter
def best_action(s, value_fn):
    """The move whose child value (from X's viewpoint) is best for the player to move:
    highest for X, lowest for O, ties to the lowest cell index. Given -- do not modify."""
    maximize = player(s) == "X"
    best_a, best_v = None, None
    for a in actions(s):
        v = value_fn(result(s, a))
        if best_v is None or (v > best_v if maximize else v < best_v):
            best_a, best_v = a, v
    return best_a


def minimax(s, stats=None):
    """Minimax value of s from X's viewpoint."""
    # TODO -- count the node, then the terminal case, then max over children for X, min for O
    return 0


def minimax_agent(s):
    # TODO -- best_action with minimax as the value function
    return None


def alphabeta(s, alpha=-math.inf, beta=math.inf, stats=None, order=None):
    """Same value as minimax(s), found with alpha-beta pruning."""
    # TODO
    return 0


def order_moves(s):
    """Legal moves: center first, then corners 0, 2, 6, 8, then edges 1, 3, 5, 7."""
    # TODO
    return actions(s)


def depth_limited(s, depth, alpha=-math.inf, beta=math.inf, stats=None):
    """Alpha-beta to `depth` plies; frontier boards are scored with evaluate(s, "X")."""
    # TODO
    return 0


def make_depth_limited_agent(depth):
    # TODO -- return an agent (a function of a state) that searches `depth` plies,
    # counting its own move as the first
    return None


# TODO -- 3.1
value_empty = None
nodes_minimax = None

# TODO -- 3.2
nodes_ab_natural = None
nodes_ab_ordered = None
ab_mismatches = None

# TODO -- 3.3 then the plot of wins and losses against depth
depth1_agreement = None
depth_sweep_df = None

Address all four:
- What is `value_empty`, and what does it say about tic-tac-toe? Is `minimax_agent` ever beaten, by anyone?  
- Compare `nodes_minimax`, `nodes_ab_natural` and `nodes_ab_ordered`. The lecture claims good ordering shrinks the effective branching factor from $b$ toward $\sqrt{b}$. Compute $n^{1/9}$ for each count as a rough effective branching factor, and say whether the claim holds up here.  
- In `depth_sweep_df`, which depths ever lose to a random player? You should see a pattern between odd and even depths. Explain it in terms of whose move the search ends on, and connect it to the horizon effect from lecture.  
- Nodes per first move grow quickly with depth, yet the win rate barely changes past depth 2. What does that say about how much this evaluation function is worth, compared with raw search depth?  

*Your answer for 3.4:*